In [1]:
import pandas as pd
df = pd.read_csv(r'F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\game_tracking_2024.csv')

In [2]:
df.columns.tolist()

['game_id',
 'play_id',
 'nfl_id',
 'time',
 'x',
 'y',
 's',
 'a',
 'dis',
 'o',
 'dir',
 'event']

In [3]:
print("Shape:", df.shape)

display(df.head())

df.info(memory_usage="deep")

Shape: (7827944, 12)


,game_id,play_id,nfl_id,time,x,y,s,a,dis,o,dir,event
0,2024080952,3379,57367,2024-08-10T01:30:31.800,87.09,22.52,0.28,0.11,0.03,98.51,126.08,NaN
1,2024080952,3379,57367,2024-08-10T01:30:31.900,87.11,22.51,0.24,0.18,0.02,97.33,125.81,NaN
2,2024080952,3379,57367,2024-08-10T01:30:32.000,87.12,22.50,0.17,0.24,0.01,96.65,124.99,NaN
3,2024080952,3379,57367,2024-08-10T01:30:32.100,87.13,22.49,0.15,0.21,0.02,95.42,120.75,NaN
4,2024080952,3379,57367,2024-08-10T01:30:32.200,87.14,22.48,0.11,0.19,0.01,95.42,136.32,NaN


<class 'pandas.DataFrame'>
RangeIndex: 7827944 entries, 0 to 7827943
Data columns (total 12 columns):
 #   Column   Dtype  
---  ------   -----  
 0   game_id  int64  
 1   play_id  int64  
 2   nfl_id   int64  
 3   time     str    
 4   x        float64
 5   y        float64
 6   s        float64
 7   a        float64
 8   dis      float64
 9   o        float64
 10  dir      float64
 11  event    str    
dtypes: float64(7), int64(3), str(2)
memory usage: 1.4 GB


In [4]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})

display(missing_summary)

,missing_count,missing_pct
game_id,0,0.00
play_id,0,0.00
nfl_id,0,0.00
time,0,0.00
x,0,0.00
y,0,0.00
s,0,0.00
a,0,0.00
dis,0,0.00
o,0,0.00


### Missing Values Analysis — game_tracking_2024

**Findings:**
- All identifier, timestamp, position, and movement columns have 0% missing values.
- The `event` column contains 7,306,249 missing values (93.34%).
- The missing percentage is nearly identical to 2023 (93.36%).

**Interpretation:**
- Missing `event` values are likely structural because events are annotated only at specific tracking timestamps.
- No missing values were detected in the remaining 11 columns.

**Preprocessing Decision:**
- Preserve all tracking observations.
- Retain missing `event` values.
- Do not automatically drop rows or impute event labels.

**Conclusion:**
The dataset is complete for identifiers and movement measurements. The high missingness in `event` is consistent with the 2023 tracking dataset.

In [5]:
candidate_key = ["game_id", "play_id", "nfl_id", "time"]

print("Full Duplicate Rows:", df.duplicated().sum())

print(
    "Duplicate Candidate Keys:",
    df.duplicated(subset=candidate_key).sum()
)

print(
    "Missing Values in Candidate Key:",
    df[candidate_key].isna().sum().sum()
)

Full Duplicate Rows: 0
Duplicate Candidate Keys: 0
Missing Values in Candidate Key: 0


### Granularity and Unique Identifier Validation — game_tracking_2024

**Granularity:** One tracking observation per player per timestamp within a game play.

**Validated Candidate Composite Key:** `(game_id, play_id, nfl_id, time)`

**Validation Results:**
- Full duplicate rows: 0
- Duplicate candidate keys: 0
- Missing values in candidate key columns: 0

**Conclusion:**
The candidate composite key uniquely identifies every tracking observation in the 2024 dataset. No duplicate records or null candidate keys were detected. Results are consistent with the 2023 tracking dataset.

In [6]:
print("Unique Games:", df["game_id"].nunique())

print(
    "Unique Plays:",
    df[["game_id", "play_id"]].drop_duplicates().shape[0]
)

print("Unique Players:", df["nfl_id"].nunique())

display(df["event"].value_counts(dropna=False).head(20))

Unique Games: 334
Unique Plays: 40331
Unique Players: 318


event
NaN                          7306249
ball_snap                     108626
pass_forward                   64826
first_contact                  64745
tackle                         64072
pass_arrived                   48877
pass_outcome_caught            39397
handoff                        31977
pass_outcome_incomplete        22030
play_action                    17841
out_of_bounds                  12225
autoevent_passinterrupted       9302
run                             6177
qb_sack                         4386
man_in_motion                   3054
line_set                        2942
play_submit                     2648
touchdown                       2604
autoevent_playaction            2348
dropped_pass                    1718
Name: count, dtype: int64

In [7]:
numeric_cols = ["x", "y", "s", "a", "dis", "o", "dir"]

display(df[numeric_cols].describe().T)

print("Negative Speed:", (df["s"] < 0).sum())
print("Negative Distance:", (df["dis"] < 0).sum())

,count,mean,std,min,25%,50%,75%,max
x,7827944.0,59.664050,24.428475,-6.33,41.37,59.54,78.00,125.66
y,7827944.0,26.773892,9.611579,-8.13,20.97,26.79,32.59,63.96
s,7827944.0,2.518076,2.075392,0.00,0.83,2.09,3.76,13.87
a,7827944.0,1.590061,1.221175,0.00,0.65,1.37,2.28,22.02
dis,7827944.0,0.255668,0.208972,0.00,0.08,0.21,0.38,16.86
o,7827944.0,178.881930,100.639108,0.00,89.25,179.97,268.71,360.00
dir,7827944.0,180.109753,103.308900,0.00,90.12,179.99,269.58,360.00


Negative Speed: 0
Negative Distance: 0


### Categorical and Numeric Validation — game_tracking_2024

**Dataset Coverage**
- Total tracking records: 7,827,944
- Unique games: 334
- Unique game-play combinations: 40,331
- Unique players: 318

**Event Distribution**
- The most frequent non-null events include `ball_snap`, `pass_forward`, `first_contact`, `tackle`, and `pass_arrived`.
- The `event` column contains 93.34% missing values, consistent with the 2023 dataset.

**Numeric Validation**
- Negative speed records: 0
- Negative distance records: 0
- Orientation and direction range from 0 to 360 degrees.
- Coordinate ranges are broadly similar to those observed in 2023.

**Potential Extreme Values**
- Maximum speed: 13.87
- Maximum acceleration: 22.02
- Maximum distance: 16.86

The maximum distance is substantially higher than typical observations and should be investigated during preprocessing.

**Comparison with 2023**
- Tracking records increased from 3,665,443 to 7,827,944.
- Unique players increased from 156 to 318.
- Unique plays increased from 31,690 to 40,331.
- Event missingness remained nearly unchanged.

**Preprocessing Decision**
- Preserve all tracking records.
- Retain missing event annotations.
- Flag extreme movement values for later investigation.
- Avoid automatic outlier removal without examining the associated tracking sequences.

**Conclusion**
The 2024 dataset passes basic numeric validity checks. Event distributions and coordinate ranges are broadly consistent with 2023, while unusually large movement values require further review.

In [9]:
games_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\games.csv")
players_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\players.csv")

print(
    "Tracking Game IDs missing from games:",
    len(set(df["game_id"]) - set(games_df["game_id"]))
)

print(
    "Tracking Player IDs missing from players:",
    len(set(df["nfl_id"]) - set(players_df["nfl_id"]))
)

missing_games_2024 = (
    set(games_df.loc[games_df["season"] == 2024, "game_id"])
    - set(df["game_id"])
)

print("2024 Games without tracking:", len(missing_games_2024))
print("Missing Game IDs:", missing_games_2024)

Tracking Game IDs missing from games: 0
Tracking Player IDs missing from players: 0
2024 Games without tracking: 0
Missing Game IDs: set()


# Data Profiling Summary — game_tracking_2024.csv

### 1. Dataset Overview
- Total records: 7,827,944
- Total columns: 12
- Unique games: 334
- Unique game-play combinations: 40,331
- Unique players: 318

### 2. Missing Values
- All identifier, timestamp, position, and movement columns have 0% missing values.
- The `event` column contains 7,306,249 missing values (93.34%).
- Missing event annotations are likely structural because events occur at specific tracking timestamps.

### 3. Granularity and Uniqueness
- Granularity: One tracking observation per player per timestamp within a game play.
- Validated candidate composite key: `(game_id, play_id, nfl_id, time)`
- Full duplicate rows: 0
- Duplicate composite keys: 0
- Missing composite key values: 0

### 4. Categorical and Numeric Validation
- Event categories are consistent with NFL gameplay.
- Negative speed records: 0
- Negative distance records: 0
- Orientation and direction values range from 0 to 360 degrees.
- Maximum speed: 13.87
- Maximum acceleration: 22.02
- Maximum distance: 16.86

**Potential Issue:** The maximum distance is unusually high relative to the overall distribution and should be reviewed during preprocessing.

### 5. Referential Integrity
- Tracking game IDs missing from `games.csv`: 0
- Tracking player IDs missing from `players.csv`: 0
- 2024 games without tracking: 0

All 334 games from the 2024 season are represented in the tracking dataset.

### 6. Recommended Preprocessing Actions
- Preserve missing `event` values.
- Retain all tracking frames.
- Investigate extreme movement measurements before applying outlier treatment.
- Use the validated composite key for frame-level uniqueness checks.
- Validate player-play coverage separately before joining with `player_play.csv`.

### Conclusion
The 2024 tracking dataset passes the core completeness, uniqueness, numeric validity, and game/player referential integrity checks.

No game-level coverage gaps were identified. Potential extreme movement measurements remain flagged for further investigation.